In [1]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

# 회귀용
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

# 머신러닝 알고리즘 - 회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import VotingRegressor

# 차원 축소
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# 군집
from sklearn.cluster import KMeans
from sklearn.cluster import MeanShift
from sklearn.cluster import estimate_bandwidth

# 학습 모델 저장을 위한 라이브러리
import pickle

# 폴더에 들어있는 파일 가져오기
import glob
import os

# 시간을 관리하는 모듈 
from datetime import datetime

# 예쁘게 출력하는 모듈
from IPython.display import display

#학습 모델 저장
import joblib

# f1 리포트 
from sklearn.metrics import classification_report, f1_score

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

## 📤데이터 불러오기

In [3]:
df_all = pd.read_csv("C:/Users/user/Desktop/workspace/14_Final_PROJECT/data/Suyeon_df_selected_only_30.csv")

## ✅ 타겟: E vs not_E 생성

In [5]:
df_all["target_1"] = df_all["Segment"].apply(lambda x: "E" if x == "E" else "not_E")

## ✅ 피처(X), 타겟(y) 분리

In [7]:
X = df_all.drop(columns=["ID", "Segment", "target_1","ID.1"]).copy()
y = df_all["target_1"]

## ✅ Label Encoding (범주형 → 숫자형)

In [9]:
for col in X.select_dtypes(include="object").columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

y_encoded = le.fit_transform(y)

## ✅ 학습/검증 분할

In [11]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, stratify=y_encoded, test_size=0.2, random_state=42
)

## ✅ 모델 학습 (XGBoost)

In [13]:
model1 = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
model1.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

## ✅ 예측

In [15]:
y_pred = model1.predict(X_val)

## ✅ 다시 숫자 → 라벨로 디코딩

In [17]:
y_val_label = le.inverse_transform(y_val)
y_pred_label = le.inverse_transform(y_pred)

## ✅ 평가 지표 출력

In [19]:
print("✅ F1 Macro:", f1_score(y_val, y_pred, average="macro"))
print("✅ F1 Micro:", f1_score(y_val, y_pred, average="micro"))
print("\n📊 Classification Report:")
print(classification_report(y_val_label, y_pred_label))

✅ F1 Macro: 0.851385633523095
✅ F1 Micro: 0.90891875

📊 Classification Report:
              precision    recall  f1-score   support

           E       0.93      0.96      0.94    384410
       not_E       0.80      0.72      0.76     95590

    accuracy                           0.91    480000
   macro avg       0.87      0.84      0.85    480000
weighted avg       0.91      0.91      0.91    480000



## 📌 Not-E 데이터 → C, D vs not_CD 분류

In [21]:
df_2 = df_all[df_all["target_1"] == "not_E"].copy()
df_2["target_2"] = df_2["Segment"].apply(lambda x: x if x in ["C", "D"] else "not_CD")
df_all["target_2"] = df_2["Segment"].apply(lambda x: x if x in ["C", "D"] else "not_CD")

## 📌 피처(X), 타겟(y) 분리

In [23]:
X_2 = df_2.drop(columns=["ID", "Segment", "ID.1", "target_1", "target_2"])
y_2 = df_2["target_2"]

## 📌 원-핫 인코딩

In [25]:
X_2_encoded = pd.get_dummies(X_2, drop_first=True)

## 📌 학습/검증 데이터 분리

In [27]:
X_train_2, X_val_2, y_train_2, y_val_2 = train_test_split(X_2_encoded, y_2, test_size=0.2, random_state=42, stratify=y_2)

## 📌 라벨 인코딩

In [29]:
le2 = LabelEncoder()
y_train_2_enc = le2.fit_transform(y_train_2)
y_val_2_enc = le2.transform(y_val_2)

## 📌 모델 학습

In [31]:
model2 = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
model2.fit(X_train_2, y_train_2_enc)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

## 📌 예측

In [33]:
y_pred_2 = model2.predict(X_val_2)

## 📌 숫자 → 라벨로 디코딩

In [35]:
y_val_2_label = le2.inverse_transform(y_val_2_enc)
y_pred_2_label = le2.inverse_transform(y_pred_2)

## 📌 평가 지표 출력

In [37]:
print("✅ F1 Macro:", f1_score(y_val_2_label, y_pred_2_label, average="macro"))
print("✅ F1 Micro:", f1_score(y_val_2_label, y_pred_2_label, average="micro"))

print("\n📊 Classification Report:")
print(classification_report(y_val_2_label, y_pred_2_label))

✅ F1 Macro: 0.7231132285185184
✅ F1 Micro: 0.8514175122920807

📊 Classification Report:
              precision    recall  f1-score   support

           C       0.77      0.63      0.69     25518
           D       0.87      0.93      0.90     69849
      not_CD       0.88      0.43      0.57       223

    accuracy                           0.85     95590
   macro avg       0.84      0.66      0.72     95590
weighted avg       0.85      0.85      0.85     95590



## 💡 Not-E 데이터 & Not-C/D 데이터 → A VS B 분류

In [39]:
df_3 = df_all[(df_all["target_1"] == "not_E") & (df_all["target_2"] == "not_CD")].copy()
df_all["target_3"] = df_3["Segment"]
df_3["target_3"] = df_3["Segment"]  # A or B 그대로 사용

## 💡 피처(X), 타겟(y) 분리

In [41]:
X_3 = df_3.drop(columns=["ID", "Segment", "ID.1", "target_1", "target_2", "target_3"])
y_3 = df_3["target_3"]

## 💡 원-핫 인코딩

In [43]:
X_3_encoded = pd.get_dummies(X_3, drop_first=True)

## 💡 학습/검증 데이터 분리

In [45]:
X_train_3, X_val_3, y_train_3, y_val_3 = train_test_split(X_3_encoded, y_3, test_size=0.2, random_state=42, stratify=y_3)

## 💡 라벨 인코딩

In [47]:
le3 = LabelEncoder()
y_train_3_enc = le3.fit_transform(y_train_3)
y_val_3_enc = le3.transform(y_val_3)

## 💡 모델 학습

In [49]:
model3 = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
model3.fit(X_train_3, y_train_3_enc)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

## 💡 예측

In [51]:
y_pred_3 = model3.predict(X_val_3)

## 💡 숫자 → 라벨로 디코딩

In [53]:
y_val_3_label = le3.inverse_transform(y_val_3_enc)
y_pred_3_label = le3.inverse_transform(y_pred_3)

## 💡 평가 지표 출력

In [55]:
print("✅ F1 Macro:", f1_score(y_val_3_label, y_pred_3_label, average="macro"))
print("✅ F1 Micro:", f1_score(y_val_3_label, y_pred_3_label, average="micro"))

print("\n📊 Classification Report:")
print(classification_report(y_val_3_label, y_pred_3_label))

✅ F1 Macro: 0.9405835543766579
✅ F1 Micro: 0.9732142857142857

📊 Classification Report:
              precision    recall  f1-score   support

           A       0.98      0.98      0.98       195
           B       0.90      0.90      0.90        29

    accuracy                           0.97       224
   macro avg       0.94      0.94      0.94       224
weighted avg       0.97      0.97      0.97       224



## 🔁 전체 파이프라인 (Train 데이터 평가용)

### 1️⃣ 전체 데이터에서 불필요한 컬럼 제거

In [58]:
X_all = df_all.drop(columns=["Segment", "ID", "ID.1", "target_1", "target_2", "target_3"], errors="ignore")

### 2️⃣ 문자열 전처리 (ex: "01.100만원+" → 01100)

In [60]:
for col in X_all.columns:
    if X_all[col].dtype == "object":
        X_all[col] = (
            X_all[col]
            .astype(str)
            .str.replace("만원", "", regex=False)
            .str.replace("+", "", regex=False)
            .str.replace(",", "", regex=False)
            .str.strip()
        )
        X_all[col] = pd.to_numeric(X_all[col], errors="coerce")

### 3️⃣ 수치형 + 범주형 처리

In [62]:
num_cols = X_all.select_dtypes(include=["int64", "float64"])
obj_cols = X_all.select_dtypes(include="object")

if not obj_cols.empty:
    cat_cols = pd.get_dummies(obj_cols, drop_first=True)
    X_all_encoded = pd.concat([num_cols, cat_cols], axis=1)
else:
    X_all_encoded = num_cols.copy()

### 4️⃣ model1 기준 컬럼 정렬

In [64]:
X_all = X_all_encoded.reindex(columns=X_train.columns, fill_value=0)

### 5️⃣ model1 → E vs not_E 예측

In [66]:
pred1 = model1.predict(X_all)
label1 = le.inverse_transform(pred1)

### 6️⃣ 예측 결과 초기화

In [68]:
final_preds = np.array(label1)

### 7️⃣ model2 → C/D vs not_CD 예측

In [70]:
idx_not_E = np.where(label1 != "E")[0]
X_not_E = X_all.iloc[idx_not_E]
X_not_E_aligned = X_not_E.reindex(columns=X_train_2.columns, fill_value=0)

pred2 = model2.predict(X_not_E_aligned)
label2 = le2.inverse_transform(pred2)

### 8️⃣ model3 → A vs B 예측 (not_CD 대상)

In [72]:
idx_not_CD = np.where(~np.isin(label2, ["C", "D"]))[0]
X_not_CD = X_not_E_aligned.iloc[idx_not_CD]
X_not_CD_aligned = X_not_CD.reindex(columns=X_train_3.columns, fill_value=0)

pred3 = model3.predict(X_not_CD_aligned)
label3 = le3.inverse_transform(pred3)

### 9️⃣ 최종 예측값 조립

In [74]:
for i, idx in enumerate(idx_not_E):
    if label2[i] in ["C", "D"]:
        final_preds[idx] = label2[i]
    else:
        # model3 결과 반영
        index_in_cd = idx_not_CD.tolist().index(i)
        final_preds[idx] = label3[index_in_cd]

### 🔟 평가

In [76]:
true_labels = df_all["Segment"]

#### 지표 확인

In [78]:
print("✅ F1 Macro:", f1_score(true_labels, final_preds, average="macro"))
print("✅ F1 Micro:", f1_score(true_labels, final_preds, average="micro"))
print("\n📊 Classification Report:")
print(classification_report(true_labels, final_preds))

✅ F1 Macro: 0.7966598611234444
✅ F1 Micro: 0.8846841666666667

📊 Classification Report:
              precision    recall  f1-score   support

           A       0.98      0.82      0.89       972
           B       0.97      0.78      0.86       144
           C       0.76      0.62      0.68    127590
           D       0.66      0.56      0.60    349242
           E       0.92      0.96      0.94   1922052

    accuracy                           0.88   2400000
   macro avg       0.86      0.75      0.80   2400000
weighted avg       0.88      0.88      0.88   2400000

